# Refua Antibody Design Notebook: PD-L1 Immune Checkpoint

> **Audience:** anyone new to antibody design workflows.  
> **Goal:** generate epitope-focused antibody candidates against a clinically validated oncology target.

## Why this target is interesting (with sources)
- **Mechanism is well validated:** PD-L1 binding to PD-1 suppresses T-cell killing, and blocking that interaction can restore antitumor activity (NCI checkpoint inhibitor overview).
- **Clinical relevance is current:** on **December 13, 2024**, FDA approved **cosibelimab-ipdl** (a PD-L1 blocking antibody) for metastatic or locally advanced cSCC.
- **Structure-guided design is practical:** high-quality human complexes exist for PD-1:PD-L1 (**4ZQK**) and PD-L1:durvalumab (**5X8M**), enabling informed epitope windows.

## Workflow
1. Define a PD-L1 antigen domain and interface-focused binding window.
2. Generate heavy/light antibody templates with explicit CDR length controls.
3. Run one design/fold pass and inspect candidate specs for triage.


In [ ]:
%load_ext refua_notebook
import refua_notebook

refua_notebook.activate()


In [ ]:
from refua import BinderDesigns, Complex, Protein


## Biological setup (plain language)

This notebook uses the human PD-L1 IgV domain from PDB 4ZQK and prioritizes the central PD-1/antibody-contact surface.

| Design input | Choice in this notebook | Why |
|---|---|---|
| Target chain | 4ZQK PD-L1 chain A sequence (115 aa) | Compact, experimentally solved human interface domain |
| Binding hotspot | `34..106` | Covers a broad PD-1/durvalumab-facing region used for checkpoint blockade designs |
| Candidate style | De novo heavy/light pair with CDR constraints | Standard early-stage antibody exploration pattern |


In [ ]:
PDL1_IGV_4ZQK = (
    "AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNA"
)

# Reference therapeutic binder sequences from PDB 5X8M (durvalumab).
DURVALUMAB_HEAVY_5X8M = (
    "EVQLVESGGGLVQPGGSLRLSCAASGFTFSRYWMSWVRQAPGKGLEWVANIKQDGSEKYYVDSVKGRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAREGGWFGELAFDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDKTHTHHHHHH"
)
DURVALUMAB_LIGHT_5X8M = (
    "EIVLTQSPGTLSLSPGERATLSCRASQRVSSSYLAWYQQKPGQAPRLLIYDASSRATGIPDRFSGSGSGTDFTLTISRLEPEDFAVYYCQQYGSLPWTFGQGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC"
)

PDL1_INTERFACE_WINDOW = "34..106"

HEAVY_CDR_LENGTHS = (12, 10, 13)
LIGHT_CDR_LENGTHS = (10, 9, 9)


## Build antibody design spec

This cell composes:
- the antigen (`A`)
- a generated heavy chain (`H`)
- a generated light chain (`L`)

into one `Complex` object named `pd_l1_checkpoint_antibody_design`.

Think of `design_spec` as a reusable design recipe you can rerun and compare across different epitope windows or CDR settings.


In [ ]:
antigen = Protein(
    PDL1_IGV_4ZQK,
    ids="A",
    binding_types={"binding": PDL1_INTERFACE_WINDOW},
)

binder_pair = BinderDesigns.antibody(
    heavy_cdr_lengths=HEAVY_CDR_LENGTHS,
    light_cdr_lengths=LIGHT_CDR_LENGTHS,
    heavy_id="H",
    light_id="L",
)

design_spec = Complex([antigen, *binder_pair], name="pd_l1_checkpoint_antibody_design")


In [ ]:
{
    "heavy_spec": binder_pair.heavy.sequence,
    "light_spec": binder_pair.light.sequence,
    "durvalumab_ref_heavy_5x8m": DURVALUMAB_HEAVY_5X8M,
    "durvalumab_ref_light_5x8m": DURVALUMAB_LIGHT_5X8M,
}


## Design narrative: from intent to candidates

This is an **early discovery pass** to generate structured antibody hypotheses.

### What each output means
| Output | What it is | How to use it |
|---|---|---|
| `result.binder_specs` | Heavy/light candidate specifications | Compare sequence hypotheses across CDR settings |
| `result.chain_design_summary()` | Per-chain summary of fixed vs designable regions | Confirm masking/constraints match your intent |
| `result.features` | Model-level features for downstream ranking workflows | Save for reproducibility and candidate triage |

> Practical strategy: run several nearby epitope windows and CDR length sets, then prioritize candidates that remain stable across settings.


## Run a design pass

`design_spec.fold()` executes the design/fold workflow for this setup.

Interpretation guardrails:
- Generated binders are **computational candidates**, not validated therapeutics.
- Use these outputs for ranking and shortlist generation.
- Confirm top candidates experimentally (binding, function, developability, safety).


In [ ]:
result = design_spec.fold()

result


In [ ]:
result.binder_specs


In [ ]:
result.chain_design_summary()


## Science references

- NCI, immune checkpoint inhibitor mechanism overview (PD-1/PD-L1): https://www.cancer.gov/about-cancer/treatment/types/immunotherapy/checkpoint-inhibitors
- FDA (December 13, 2024), cosibelimab-ipdl approval for advanced cSCC: https://www.fda.gov/drugs/resources-information-approved-drugs/fda-approves-cosibelimab-ipdl-metastatic-or-locally-advanced-cutaneous-squamous-cell-carcinoma
- RCSB PDB 4ZQK (human PD-1:PD-L1 complex): https://www.rcsb.org/structure/4ZQK
- RCSB PDB 5X8M (PD-L1 in complex with durvalumab): https://www.rcsb.org/structure/5X8M
- Zak et al., *PNAS* (2015), PD-1/PD-L1 complex structure: https://www.pnas.org/doi/10.1073/pnas.1500472112
- FDA preclinical research basics (why computational designs need wet-lab validation): https://www.fda.gov/patients/drug-development-process/step-2-preclinical-research
